# Export Invoice Summary — Data Processing Pipeline

This notebook pulls invoice data (Woven and Sweater company groups) from the
reporting database, cleans and standardizes it, and produces the following
deliverables:

1. **EIS.xlsx** — the master, cleaned Export Invoice Summary
2. **RealizeReports/** — Realize Pending report, split by owner/category
3. **OnBoardPending.xlsx** — invoices awaiting on-board status
4. **ExFactoryPending.xlsx** — invoices awaiting ex-factory shipment
5. **BankSubmitPending.xlsx** — invoices awaiting bank submission

## 1. Setup & Configuration

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import pyodbc
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# Output locations
# ----------------------------------------------------------------------
OUTPUT_DIR = "OUTPUT"
REALIZE_REPORTS_DIR = os.path.join(OUTPUT_DIR, "RealizeReports")

for directory in (OUTPUT_DIR, REALIZE_REPORTS_DIR):
    os.makedirs(directory, exist_ok=True)

## 2. Database Connection

In [2]:
load_dotenv()

DB_SERVER = os.getenv("DB_SERVER")
DB_PORT = os.getenv("DB_PORT")
DB_DATABASE = os.getenv("DB_DATABASE")
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")

CONN_STR = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={DB_SERVER},{DB_PORT};"
    f"DATABASE={DB_DATABASE};"
    f"UID={DB_USERNAME};"
    f"PWD={DB_PASSWORD};"
    "TrustServerCertificate=yes;"  # stability improvement
    "Encrypt=no;"                  # stability improvement
    "MARS_Connection=yes;"         # required for multi-result stored procedures
)


def get_connection():
    """Open a SQL Server connection using the configured credentials.

    Returns
    -------
    pyodbc.Connection | None
        An open, autocommitting connection, or ``None`` if the connection
        attempt failed (the error is printed for diagnostics).
    """
    try:
        conn = pyodbc.connect(CONN_STR, timeout=10, autocommit=True)
        conn.timeout = 30  # query execution timeout (seconds)
        print("Connected successfully")
        return conn
    except pyodbc.Error as e:
        print("Connection failed:", e)
        return None

In [3]:
conn = get_connection()

if conn is None:
    print("CRITICAL: Connection could not be established. Check DB_SERVER and DB_PASSWORD.")

Connected successfully


## 3. Extract Invoice Data

Data is pulled separately for the two company groups (Woven = 1, Sweater = 2)
via the `Report_ExportInvoiceSummary` stored procedure, then combined into a
single working `DataFrame`.

In [4]:
def fetch_invoice_summary(connection, company_group_id, from_date, to_date,
                           report_type_id=0, fiscal_year_id=0, is_realize=0):
    """Run the Report_ExportInvoiceSummary stored procedure and return the result.

    Parameters
    ----------
    connection : pyodbc.Connection
        An open database connection.
    company_group_id : int
        1 = Woven, 2 = Sweater.
    from_date, to_date : str
        Date range in 'YYYY-MM-DD' format.

    Returns
    -------
    pandas.DataFrame
    """
    query = f"""
    SET NOCOUNT ON;
    EXEC Report_ExportInvoiceSummary
         @ReportTypeID   = {report_type_id},
         @FiscalYearID   = {fiscal_year_id},
         @FromDate       = '{from_date}',
         @ToDate         = '{to_date}',
         @CompanyGroupID = {company_group_id},
         @isRealize      = {is_realize}
    """
    return pd.read_sql_query(query, connection)

In [5]:
if conn is not None:
    # Woven = CompanyGroupID 1
    woven = fetch_invoice_summary(
        conn, company_group_id=1, from_date="2020-01-01", to_date="2026-12-31"
    )
    print(f"Woven Data Shape   : {woven.shape}")

    # Sweater = CompanyGroupID 2
    sweater = fetch_invoice_summary(
        conn, company_group_id=2, from_date="2020-01-01", to_date="2026-12-31"
    )
    print(f"Sweater Data Shape : {sweater.shape}")
else:
    raise RuntimeError("Connection is None. Check your credentials/server.")

OperationalError: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 18 for SQL Server]Query timeout expired (0) (SQLExecDirectW)')

In [ ]:
df = pd.concat([woven, sweater], ignore_index=False)
print(f"Combined Data Shape : {df.shape}")
df.sample(5)

Combined Data Shape : (51613, 44)


,LCFactory,LCNo,LCValue,LCQuantity,PaymentTerm,SalesInvoice,ExpNo,ExpDate,SalesInvoiceDate,Buyer,...,BankRefNo,BankSubmitDate,IsFDBC,BankShortName,ShippingModeName,InvoiceStatus,RealizeNO,RealizeDate,RealizeValue,IsRealize
35868,MGSL,MGSL/Women Shirt S.2,2995246.39,619756.0,EOM+63,2509083,00001689-020351-2025,2025-07-17,2025-07-16 11:44:39.097,H&M,...,FDC-4510-25,2025-11-30,Yes,DBBL,Sea,Incentive Pending,DR-11-25-062,30-Nov-2025,321.42,Yes
49748,MGL,SC-2023-CCL0000118,407069.80,429864.0,At Sight,2609754,00000947-001561-2026,2026-08-25,2026-08-25 00:00:00.000,Primark,...,NaN,NaT,No,NBL,Sea,On Board Pending,NaN,NaN,NaN,No
45889,MGSL,MGSL/H&M-Mens DBL-S.04,8891547.77,2304529.0,EOM+63,2605632,00001689-012624-2026,2026-05-17,2026-05-17 00:00:00.000,H&M,...,FDBC-2879-26,2026-08-03,Yes,DBBL,Sea,Incentive Pending,DR-08-26-017,03-Aug-2026,2059.95,Yes
49343,MGSL,MGSL/H&M-Mens DBL-S.04,8891547.77,2304529.0,EOM+63,2609315,00001689-020114-2026,2026-08-16,2026-08-16 14:11:19.260,H&M,...,NaN,NaT,No,DBBL,Sea,Bank Submit Pending,NaN,NaN,NaN,No
7848,MGSL,MGSL/PRINCETON SHIRT - 8757 S.08,3204657.14,761705.0,60 Days,MGSL/23/3236,005892,2023-05-01,2023-05-01 00:00:00.000,H&M,...,502/23,2023-06-07,Yes,ABL,Sea,Incentive Pending,DR-06-23-004,07-Jun-2023,31.90,Yes


In [ ]:
eis_path = os.path.join(OUTPUT_DIR, "ExportInvoiceSummary_ROW.xlsx")
df.to_excel(eis_path, index=False)
print(f"Saved EIS to: {eis_path}")

Saved EIS to: OUTPUT\ExportInvoiceSummary_ROW.xlsx


In [ ]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')>

### Find information

In [ ]:
## search for a specific invoice number in the combined DataFrame

search_term = "2606243"

if "woven" in globals() and "sweater" in globals():
    raw_df = pd.concat([woven, sweater], ignore_index=True)

    if "SalesInvoice" in raw_df.columns:
        matches = raw_df[
            raw_df["SalesInvoice"].astype(str).str.contains(search_term, case=False, na=False)
        ]
    else:
        matches = pd.DataFrame()
elif "SalesInvoice" in df.columns:
    matches = df[
        df["SalesInvoice"].astype(str).str.contains(search_term, case=False, na=False)
    ]
elif "Invoice_No" in df.columns:
    matches = df[
        df["Invoice_No"].astype(str).str.contains(search_term, case=False, na=False)
    ]
else:
    matches = pd.DataFrame()

matches

,LCFactory,LCNo,LCValue,LCQuantity,PaymentTerm,SalesInvoice,ExpNo,ExpDate,SalesInvoiceDate,Buyer,...,BankRefNo,BankSubmitDate,IsFDBC,BankShortName,ShippingModeName,InvoiceStatus,RealizeNO,RealizeDate,RealizeValue,IsRealize
51490,MGKFL,786LC32614325,215721.55,33057.0,90 Days,2606243,00001689-013881-2026,2026-06-04,2026-06-03,s.Oliver Bernd Freier GmbH and Co.KG,...,82671.90,2026-06-15,Yes,DBBL,Sea,Realize Pending,NaN,NaN,NaN,No


In [ ]:
# Shuffle the combined records
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Sample of Shuffled Data:")
df.sample(10)

Sample of Shuffled Data:


,LCFactory,LCNo,LCValue,LCQuantity,PaymentTerm,SalesInvoice,ExpNo,ExpDate,SalesInvoiceDate,Buyer,...,BankRefNo,BankSubmitDate,IsFDBC,BankShortName,ShippingModeName,InvoiceStatus,RealizeNO,RealizeDate,RealizeValue,IsRealize
11576,MGSL,MGSL/H&M-MENS -S.07,2948973.12,763766.0,EOM 63,2313159,042/013642/23,2023-09-10,2023-09-10 00:00:00.000,H&M,...,FDBC-944/23,2023-10-08,Yes,ABL,Sea,Incentive Pending,DR-10-23-035,08-Oct-2023,48.48,Yes
39246,MGSL,MGSL/H&M-MENS-S.0,7963192.82,1672612.0,EOM+63,2407834,00000042-012786-2024,2024-05-26,2024-05-25 00:00:00.000,H&M,...,FDBC-699/24 (AGRANI PART),2024-06-06,Yes,ABL,Sea,Incentive Pending,DR-06-24-006,08-Jun-2024,91.77,Yes
8383,MGSL,MGSL/H&M Kids S2,11273220.28,3616545.0,EOM 63,2602752,00001689-006178-2026,2026-03-01,2026-03-01 10:13:17.830,H&M,...,FDBC-1414/26,2026-04-07,Yes,DBBL,Sea,Incentive Pending,DR-04-26-006,07-Apr-2026,34.32,Yes
48185,MGSL,MGSL/AMANDA SHIRT S.2,863784.15,213170.0,EOM+63,2504304,00001689-009418-2025,2025-03-17,2025-03-16 19:12:41.590,H&M,...,FDC-1900-25,2025-05-15,Yes,DBBL,Sea,Incentive Pending,DR-05-25-031,15-May-2025,24713.17,Yes
11909,MGNSL,LI24J02495,55500.00,5550.0,L/C - 75 days,2506848,00000042-003351-2025,2025-05-20,2025-05-18 00:00:00.000,DK COMPANY CPH A/S,...,FDBC/0598/25,2025-06-02,Yes,ABL,Sea,Incentive Pending,DR-06-26-031,10-Jun-2026,19050.00,Yes
27810,MGSL,MGSL/H&M-MENS-S.09,4542005.37,1244813.0,EOM+63,2404731,042/008096/24,2024-03-24,2024-03-24 00:00:00.000,H&M,...,529/24,2024-04-30,Yes,ABL,Sea,Incentive Pending,DR-05-24-047,08-May-2024,30.70,Yes
28184,MGSL,MGSL/H&M Kids DBBL-S.0,7767885.39,2760068.0,EOM+63,2414036,00001689-017630-2024,2024-09-24,2024-09-24 00:00:00.000,H&M,...,3931/24,2024-11-07,Yes,DBBL,Sea,Incentive Pending,DR-11-24-004,07-Nov-2024,5486.88,Yes
2448,MGSL,MGSL/H&M-MENS -S.07,2948973.12,763766.0,EOM 63,2313144,042/013627/23,2023-09-10,2023-09-10 00:00:00.000,H&M,...,FDBC-943/23,2023-10-05,Yes,ABL,Sea,Incentive Pending,DR-10-23-036,05-Oct-2023,466.64,Yes
2842,MGSL,MGSL/Women Shirt S.1,1921510.96,393613.0,EOM+63,2416770,00001689-023392-2024,2024-11-17,2024-11-17 11:48:29.723,H&M,...,FDC-0180-25,2025-01-12,Yes,DBBL,Sea,Incentive Pending,DR-01-25-035,12-Jan-2025,1482.00,Yes
41526,MGSL,MGSL/H&M Kids S2,11273220.28,3616545.0,EOM 63,2603086,00001689-006987-2026,2026-03-09,2026-03-08 12:11:47.853,H&M,...,FDBC-1769/26,2026-05-05,Yes,DBBL,Sea,Incentive Pending,DR-05-26-006,06-May-2026,1026.76,Yes


## 4. Initial Data Quality Check

In [ ]:
df.isnull().sum()

LCFactory                 0
LCNo                      0
LCValue                   0
LCQuantity                0
PaymentTerm             181
SalesInvoice              0
ExpNo                    76
ExpDate                  76
SalesInvoiceDate          0
Buyer                     0
OrderNo                  15
Merchandiser            170
PortName                  3
OrderQty                  0
Dozen                     0
ctnQty                 1117
TotalFOB                  0
DiscountAmountUSD         0
CommissionAmountFC        0
ExFactoryQty            206
InvExFactoryDate        200
ExfactoryDate             0
ExFactoryNo             206
ExFactory               206
OnBoardDate             850
DueDate                1028
FVesselNo             51241
MVesselNo             13728
ShippingBillNo         1202
SBillDate              8209
BLNo                    804
BLDate                 3675
FCRNo                  4000
FCRDate                4062
BankRefNo              3756
BankSubmitDate      

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51320 entries, 0 to 51319
Data columns (total 44 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   LCFactory           51320 non-null  str           
 1   LCNo                51320 non-null  str           
 2   LCValue             51320 non-null  float64       
 3   LCQuantity          51320 non-null  float64       
 4   PaymentTerm         51139 non-null  str           
 5   SalesInvoice        51320 non-null  str           
 6   ExpNo               51244 non-null  str           
 7   ExpDate             51244 non-null  datetime64[us]
 8   SalesInvoiceDate    51320 non-null  datetime64[us]
 9   Buyer               51320 non-null  str           
 10  OrderNo             51305 non-null  str           
 11  Merchandiser        51150 non-null  str           
 12  PortName            51317 non-null  str           
 13  OrderQty            51320 non-null  float64       
 14  D

## 5. Data Cleaning & Type Casting

Raw values coming from the database/report are not always clean (mixed
date formats, currency symbols, thousands separators, literal "null"
strings, etc.). The helpers below parse each column defensively and fall
back to the original value whenever a clean conversion isn't possible, so
no data is silently lost.

In [ ]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')>

In [ ]:
# # ----------------------------------------------------------------------
# # Column definitions (raw, pre-rename, column names as returned by the DB)
# # ----------------------------------------------------------------------
# date_columns = [
#     "ExpDate", "SalesInvoiceDate", "InvExFactoryDate", "ExfactoryDate", "OnBoardDate",
#     "DueDate", "SBillDate", "BLDate", "FCRDate", "BankSubmitDate", "RealizeDate",
# ]



In [ ]:
df.columns

Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')

In [ ]:
# def safe_to_datetime(series):
#     """Parse a Series of dates, tolerating junk values and mixed formats."""
#     s = series.astype(str).str.strip()
#     s = s.replace(["", "nan", "None", "-", "--", "null", "NaT"], np.nan)

#     parsed = pd.to_datetime(s, errors="coerce", dayfirst=True)
#     fallback = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")

#     return parsed.fillna(fallback)

# def find_bad_dates(frame, col):
#     """Debug helper: return the distinct values in `col` that failed date parsing."""
#     mask = pd.to_datetime(frame[col], errors="coerce").isna() & frame[col].notna()
#     return frame.loc[mask, col].unique()

In [ ]:
# # Convert date columns (preserving the original value where parsing fails)
# for col in date_columns:
#     if col in df.columns:
#         original = df[col].copy()
#         converted = safe_to_datetime(df[col])
#         df[col] = converted.where(converted.notna(), original)

# Convert integer columns

numeric_int_columns = ["LCQuantity", "OrderQty", "ctnQty", "Dozen", "ctnQty","ExFactoryQty"]

numeric_float_columns = [
    "LCValue", "Dozen", "TotalFOB", "DiscountAmountUSD", "CommissionAmountFC", "RealizeValue", "TotalFOB", "DiscountAmountUSD", "CommissionAmountFC"]

def safe_to_numeric(series, is_int=False):
    """Parse a Series of numbers, stripping currency symbols/commas and junk values."""
    s = series.astype(str).str.strip()
    s = s.str.replace(",", "", regex=False)
    s = s.str.replace("৳", "", regex=False)
    s = s.str.replace("$", "", regex=False)
    s = s.replace(["", "nan", "None", "-", "--", "null"], np.nan)

    num = pd.to_numeric(s, errors="coerce")
    return num.round().astype("Int64") if is_int else num



for col in numeric_int_columns:
    if col in df.columns:
        original = df[col].copy()
        converted = safe_to_numeric(df[col], is_int=True)
        try:
            df[col] = converted.where(converted.notna(), original)
        except (TypeError, ValueError):
            # A nullable Int64 column can't hold a non-numeric fallback
            # value (e.g. a literal "null"/"-" placeholder from the source
            # system). Degrade to plain objects instead of crashing, which
            # matches how the date/float columns above already behave.
            df[col] = converted.astype(object).where(converted.notna(), original)

# Convert float columns
for col in numeric_float_columns:
    if col in df.columns:
        original = df[col].copy()
        converted = safe_to_numeric(df[col], is_int=False)
        df[col] = converted.where(converted.notna(), original)

# Clean text columns
text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(["nan", "None", "null", ""], np.nan)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51320 entries, 0 to 51319
Data columns (total 44 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   LCFactory           51320 non-null  str           
 1   LCNo                51320 non-null  str           
 2   LCValue             51320 non-null  float64       
 3   LCQuantity          51320 non-null  Int64         
 4   PaymentTerm         51139 non-null  str           
 5   SalesInvoice        51320 non-null  str           
 6   ExpNo               51160 non-null  str           
 7   ExpDate             51244 non-null  datetime64[us]
 8   SalesInvoiceDate    51320 non-null  datetime64[us]
 9   Buyer               51320 non-null  str           
 10  OrderNo             51305 non-null  str           
 11  Merchandiser        51150 non-null  str           
 12  PortName            51317 non-null  str           
 13  OrderQty            51320 non-null  Int64         
 14  D

In [ ]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')>

In [ ]:
# date_columns = [col for col in df.columns if "Date" in col]

# for col in date_columns:
#     original = df[col].copy()
#     converted = pd.to_datetime(original, errors="coerce", dayfirst=True).dt.normalize()
#     df[col] = converted.where(converted.notna(), original)

# df.info()

In [ ]:
df.columns

Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')

## 6. Rename Columns to Standardized Names

In [ ]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')>

In [ ]:
# ============================================================
# Rename columns to readable and standardized names
# ============================================================

df = df.rename(columns={

    # LC Information
    "LCFactory": "LC_Factory",
    "LCNo": "LC_No",
    "LCValue": "LC_Value",
    "LCQuantity": "LC_Qty",
    "PaymentTerm": "LC_Payment_Term",

    # Invoice Information
    "SalesInvoice": "Invoice_No",
    "SalesInvoiceDate": "Invoice_Date",
    "OrderQty": "Invoice_Qty",
    "TotalFOB": "Invoice_Value",

    # Export Information
    "ExpNo": "Exp_No",
    "ExpDate": "Exp_Date",

    # Buyer and Order Information
    "Buyer": "Buyer",
    "OrderNo": "Order_No",
    "Merchandiser": "Merchandiser",
    "PortName": "Port",

    # Quantity and Financial Information
    "ctnQty": "Ctn_Qty",
    "Dozen": "Dozen",
    "DiscountAmountUSD": "Discount",
    "CommissionAmountFC": "Commission",

    # Ex-Factory Information
    "ExFactoryNo": "ExFactory_No",
    "ExFactory": "ExFactory",
    "ExFactoryQty": "ExFactory_Qty",
    "ExfactoryDate": "ExFactory_Date",
    "InvExFactoryDate": "Last_ExFactory_Date",

    # Shipment Information
    "ShippingBillNo": "Shipping_Bill_No",
    "SBillDate": "Shipping_Bill_Date",
    "ShippingModeName": "Shipping_Mode",
    "OnBoardDate": "OnBoard_Date",

    # Vessel Information
    "FVesselNo": "FVessel_No",
    "MVesselNo": "MVessel_No",

    # BL Information
    "BLNo": "BL_No",
    "BLDate": "BL_Date",

    # FCR Information
    "FCRNo": "FCR_No",
    "FCRDate": "FCR_Date",

    # Bank Information
    "BankRefNo": "FDBC_No",
    "BankSubmitDate": "BankSubmit_Date",
    "IsFDBC": "Is_FDBC?",
    "BankShortName": "Bank",

    # Invoice Status
    "InvoiceStatus": "Invoice_Status",

    # Realization Information
    "RealizeNO": "Realize_No",
    "RealizeDate": "Realize_Date",
    "RealizeValue": "Realize_Value",
    "IsRealize": "Is_Realize?",

    # Due Date
    "DueDate": "Due_Date",
})


# Display final column names
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['LC_Factory', 'LC_No', 'LC_Value', 'LC_Qty', 'LC_Payment_Term',
       'Invoice_No', 'Exp_No', 'Exp_Date', 'Invoice_Date', 'Buyer', 'Order_No',
       'Merchandiser', 'Port', 'Invoice_Qty', 'Dozen', 'Ctn_Qty',
       'Invoice_Value', 'Discount', 'Commission', 'ExFactory_Qty',
       'Last_ExFactory_Date', 'ExFactory_Date', 'ExFactory_No', 'ExFactory',
       'OnBoard_Date', 'Due_Date', 'FVessel_No', 'MVessel_No',
       'Shipping_Bill_No', 'Shipping_Bill_Date', 'BL_No', 'BL_Date', 'FCR_No',
       'FCR_Date', 'FDBC_No', 'BankSubmit_Date', 'Is_FDBC?', 'Bank',
       'Shipping_Mode', 'Invoice_Status', 'Realize_No', 'Realize_Date',
       'Realize_Value', 'Is_Realize?'],
      dtype='str')>

## 7. Derive the `Category` Column

In [ ]:
# Factories MFSL / MGKFL belong to the Sweater category; everything else is Woven
df["Category"] = np.where(df["LC_Factory"].isin(["MFSL", "MGKFL"]), "Sweater", "Woven")

## 8. Finalize the Master Invoice Table

Adds a serial column, formats every date column as `DD-MM-YYYY` text for
reporting purposes, and arranges columns into the agreed report order.

In [ ]:
import pandas as pd

# ============================================================
# 1. Remove existing Serial Number column and regenerate it
# ============================================================

# Remove all existing SL# columns, including duplicate SL# columns
df = df.loc[:, df.columns != "SL#"].copy()

# Insert a fresh serial number column at the beginning
df.insert(0, "SL#", range(1, len(df) + 1))


# ============================================================
# 2. Format date columns as DD-MM-YYYY
# ============================================================


# <bound method IndexOpsMixin.tolist of Index(['LC_Factory', 'LC_No', 'LC_Value', 'LC_Qty', 'LC_Payment_Term',
#        'Invoice_No', 'Exp_No', 'Exp_Date', 'Invoice_Date', 'Buyer', 'Order_No',
#        'Merchandiser', 'Port', 'Invoice_Qty', 'Dozen', 'Ctn_Qty',
#        'Invoice_Value', 'Discount', 'Commission', 'ExFactory_Qty',
#        'Last_ExFactory_Date', 'ExFactory_Date', 'ExFactory_No', 'ExFactory',
#        'OnBoard_Date', 'Due_Date', 'FVessel_No', 'MVessel_No',
#        'Shipping_Bill_No', 'Shipping_Bill_Date', 'BL_No', 'BL_Date', 'FCR_No',
#        'FCR_Date', 'FDBC_No', 'BankSubmit_Date', 'Is_FDBC?', 'Bank', 'Mode',
#        'Invoice_Status', 'Realize_No', 'Realize_Date', 'Realize_Value',
#        'Is_Realize?', 'Category'],
#       dtype='str')>

# Date columns that should be displayed as DD-MM-YYYY
date_display_columns = [
    "Invoice_Date",
    "Exp_Date",
    "ExFactory_Date",
    "Last_ExFactory_Date",
    "Shipping_Bill_Date",
    "OnBoard_Date",
    "BL_Date",
    "Due_Date",
    "FCR_Date",
    "BankSubmit_Date",
    "Realize_Date",
]

# Format only the date columns that actually exist in the DataFrame
for col in date_display_columns:

    if col in df.columns:

        # Convert to datetime safely
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce",
            dayfirst=True
        ).dt.strftime("%d-%m-%Y")

        # Replace NaN/NaT-generated values with blank
        df[col] = df[col].fillna("")


# ============================================================
# 3. Define the required master report column order
# ============================================================

desired_order = [
    "SL#",
    "Category",

    "LC_Factory",
    "LC_No",
    "LC_Value",
    "LC_Qty",
    "LC_Payment_Term",

    "Invoice_No",
    "Invoice_Date",
    "Invoice_Value",
    "Invoice_Qty",

    "Exp_No",
    "Exp_Date",

    "Buyer",
    "Merchandiser",
    "Port",

    "Ctn_Qty",
    "Discount",
    "Commission",
    "Dozen",

    "ExFactory_No",
    "ExFactory",
    "ExFactory_Date",
    "ExFactory_Qty",
    "Last_ExFactory_Date",

    "Shipping_Bill_No",
    "Shipping_Bill_Date",
    "Shipping_Mode",
    "OnBoard_Date",

    "Fvessel_No",
    "MVessel_No",

    "BL_No",
    "BL_Date",
    "Due_Date",

    "FCR_No",
    "FCR_Date",

    "Bank",
    "FDBC_No",
    "BankSubmit_Date",
    "Is_FDBC?",

    "Invoice_Status",

    "Realize_No",
    "Realize_Date",
    "Realize_Value",
    "Is_Realize?",

    "Order_No",
]


# ============================================================
# 4. Reorder columns safely
# ============================================================

# Columns from desired_order that exist in the DataFrame
existing_columns = [
    col for col in desired_order
    if col in df.columns
]


# Optional: Keep any additional columns that are not in desired_order
extra_columns = [
    col for col in df.columns
    if col not in existing_columns
]


# Final DataFrame
df = df[existing_columns + extra_columns]


# ============================================================
# 5. Final validation
# ============================================================

# Reset index after rearranging columns
df.reset_index(drop=True, inplace=True)


# Display final column names
print("Final DataFrame Shape:", df.shape)
print("\nFinal Columns:")
df.columns

Final DataFrame Shape: (51320, 46)

Final Columns:


Index(['SL#', 'Category', 'LC_Factory', 'LC_No', 'LC_Value', 'LC_Qty',
       'LC_Payment_Term', 'Invoice_No', 'Invoice_Date', 'Invoice_Value',
       'Invoice_Qty', 'Exp_No', 'Exp_Date', 'Buyer', 'Merchandiser', 'Port',
       'Ctn_Qty', 'Discount', 'Commission', 'Dozen', 'ExFactory_No',
       'ExFactory', 'ExFactory_Date', 'ExFactory_Qty', 'Last_ExFactory_Date',
       'Shipping_Bill_No', 'Shipping_Bill_Date', 'Shipping_Mode',
       'OnBoard_Date', 'MVessel_No', 'BL_No', 'BL_Date', 'Due_Date', 'FCR_No',
       'FCR_Date', 'Bank', 'FDBC_No', 'BankSubmit_Date', 'Is_FDBC?',
       'Invoice_Status', 'Realize_No', 'Realize_Date', 'Realize_Value',
       'Is_Realize?', 'Order_No', 'FVessel_No'],
      dtype='str')

In [ ]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['SL#', 'Category', 'LC_Factory', 'LC_No', 'LC_Value', 'LC_Qty',
       'LC_Payment_Term', 'Invoice_No', 'Invoice_Date', 'Invoice_Value',
       'Invoice_Qty', 'Exp_No', 'Exp_Date', 'Buyer', 'Merchandiser', 'Port',
       'Ctn_Qty', 'Discount', 'Commission', 'Dozen', 'ExFactory_No',
       'ExFactory', 'ExFactory_Date', 'ExFactory_Qty', 'Last_ExFactory_Date',
       'Shipping_Bill_No', 'Shipping_Bill_Date', 'Shipping_Mode',
       'OnBoard_Date', 'MVessel_No', 'BL_No', 'BL_Date', 'Due_Date', 'FCR_No',
       'FCR_Date', 'Bank', 'FDBC_No', 'BankSubmit_Date', 'Is_FDBC?',
       'Invoice_Status', 'Realize_No', 'Realize_Date', 'Realize_Value',
       'Is_Realize?', 'Order_No', 'FVessel_No'],
      dtype='str')>

In [ ]:
df.columns

Index(['SL#', 'Category', 'LC_Factory', 'LC_No', 'LC_Value', 'LC_Qty',
       'LC_Payment_Term', 'Invoice_No', 'Invoice_Date', 'Invoice_Value',
       'Invoice_Qty', 'Exp_No', 'Exp_Date', 'Buyer', 'Merchandiser', 'Port',
       'Ctn_Qty', 'Discount', 'Commission', 'Dozen', 'ExFactory_No',
       'ExFactory', 'ExFactory_Date', 'ExFactory_Qty', 'Last_ExFactory_Date',
       'Shipping_Bill_No', 'Shipping_Bill_Date', 'Shipping_Mode',
       'OnBoard_Date', 'MVessel_No', 'BL_No', 'BL_Date', 'Due_Date', 'FCR_No',
       'FCR_Date', 'Bank', 'FDBC_No', 'BankSubmit_Date', 'Is_FDBC?',
       'Invoice_Status', 'Realize_No', 'Realize_Date', 'Realize_Value',
       'Is_Realize?', 'Order_No', 'FVessel_No'],
      dtype='str')

## 9. Export the Master Report

In [ ]:
df.to_excel(os.path.join(OUTPUT_DIR, "EIS.xlsx"), index=False)

## 10. Pending Data Extraction

In [ ]:
# Buyer_Payment = pd.read_csv(os.path.join("Data", "Buyer_Payment_Chart .csv"))
Buyer_Payment = pd.read_csv('Data/Buyer_Payment_Chart.csv')
Buyer_Payment.sample(5)

,Sl. No.,Buyer,Export Responsible,Ex-Factory,Days,Onboard,Days.1,B/L Collection,Days.2,Bank Doc Submission,Days.3,Payment Term,Payment Date,Remarks,Payment After Ex-Factory
5,6,Auric Sourcing BD,NaN,01-Jan-26,7.0,08-Jan-26,5.0,13-Jan-26,NaN,NaN,NaN,45,27-Feb-26,not update,112
118,119,MOTION DESIGN AND FASHION MART,NaN,01-Jan-26,7.0,08-Jan-26,3.0,11-Jan-26,NaN,NaN,NaN,30,10-Feb-26,not update,80
41,42,Fashion Fleet Clothing,NaN,01-Jan-26,7.0,08-Jan-26,3.0,11-Jan-26,3.0,NaN,NaN,50,02-Mar-26,not update,120
7,8,A.M.London Fashion Ltd,Qutubuddin,01-Jan-26,7.0,08-Jan-26,3.0,11-Jan-26,NaN,NaN,NaN,60,12-Mar-26,NaN,140
91,92,SIGRA VENTURES LIMITED,NaN,01-Jan-26,7.0,08-Jan-26,3.0,11-Jan-26,NaN,NaN,NaN,20,31-Jan-26,not update,60


In [ ]:
# Merge in the buyer's payment term, keeping all rows from the master table
df = df.merge(
    Buyer_Payment[["Buyer", "Payment After Ex-Factory"]],
    on="Buyer",
    how="left",
)
df = df.rename(columns={"Payment After Ex-Factory": "PaymentAfterExFactory"})

In [ ]:
# Move the new column next to Invoice_No
cols = df.columns.tolist()
if "PaymentAfterExFactory" in cols and "Invoice_No" in cols:
    cols.insert(cols.index("Invoice_No"), cols.pop(cols.index("PaymentAfterExFactory")))
    df = df[cols]
else:
    raise KeyError("Missing required columns for reorder")

df = df.rename(columns={"PaymentAfterExFactory": "Payment_After_ExFactory"})
df.sample(4)

,SL#,Category,LC_Factory,LC_No,LC_Value,LC_Qty,LC_Payment_Term,Payment_After_ExFactory,Invoice_No,Invoice_Date,...,FDBC_No,BankSubmit_Date,Is_FDBC?,Invoice_Status,Realize_No,Realize_Date,Realize_Value,Is_Realize?,Order_No,FVessel_No
6361,6361,Woven,MGSL,MGSL/H&M-MENS-S.09,4542005.37,1244813,EOM+63,22.0,2400264,04-01-2024,...,157/24,06-02-2024,Yes,Incentive Pending,DR-02-24-030,07-02-2024,2604.70,Yes,847929-8617,NaN
41769,41765,Sweater,MFSL,MFSL-LAG-23-321,159190.80,55072,At Sight,80.0,2402986,19-02-2024,...,FDBC-737/24,19-03-2024,Yes,Incentive Pending,DR-03-24-008,19-03-2024,48567.54,Yes,22891,NaN
40698,40694,Woven,MGSL,MGSL/H&M-Mens DBL-S.02,12435079.87,3145397,EOM 63,22.0,2508851,10-07-2025,...,FDC-4373-25,19-11-2025,Yes,Incentive Pending,DR-11-25-050,19-11-2025,2308.34,Yes,440980-8617,NaN
40524,40520,Woven,MGSL,MGSL/Women Shirt S.1,1921510.96,393613,EOM+63,22.0,2504875,23-03-2025,...,FDC-2211-25,04-06-2025,Yes,Incentive Pending,DR-06-25-022,04-06-2025,251.43,Yes,366958-1515,NaN


## 11. Normalize Column Names

A one-time defensive pass (whitespace-stripping, de-duplication) applied
once here rather than repeated inside every report builder below.

In [ ]:
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.duplicated()]
df["Invoice_Status"].unique()

<ArrowStringArray>
[  'Incentive Pending', 'Bank Submit Pending',    'On Board Pending',
         'For Booking',     'Realize Pending',   'ExFactory Pending']
Length: 6, dtype: str

## 12. Realize Pending Report

In [ ]:
# def build_realize_pending_report(source_df):
#     """Summarize 'Realize Pending' invoices by bank reference number (FDBC_No).

#     One output row is produced per FDBC_No, aggregating invoice totals, the
#     on-board date range, the calculated due date, and a realizability flag.
#     """
#     realize_report = source_df[
#         source_df["Invoice_Status"].astype(str).str.strip().eq("Realize Pending")
#     ].copy()

#     # Parse date columns needed for this report (kept as real datetimes for calculation)
#     date_cols = [
#         "Invoice_Date", "Exp_Date", "ExFactory_Date", "OnBoard_Date",
#         "BankSubmit_Date", "Realize_Date", "Due_Date",
#     ]
#     for col in date_cols:
#         if col in realize_report.columns:
#             realize_report[col] = pd.to_datetime(
#                 realize_report[col], errors="coerce", dayfirst=True
#             )

#     # Numeric clean-up
#     def extract_number(series):
#         return pd.to_numeric(
#             series.astype(str).str.extract(r"(\d+\.?\d*)", expand=False),
#             errors="coerce",
#         )

#     realize_report["Invoice_Value"] = pd.to_numeric(
#         realize_report["Invoice_Value"], errors="coerce"
#     ).fillna(0)
#     realize_report["Payment_After_ExFactory"] = extract_number(
#         realize_report["Payment_After_ExFactory"]
#     )

#     # Normalize the text fields used for grouping
#     for col in ["LC_Payment_Term", "FDBC_No"]:
#         if col not in realize_report.columns:
#             continue
#         realize_report[col] = realize_report[col].astype("string").str.strip()
#         realize_report.loc[
#             realize_report[col].str.lower().isin(["nan", "none", "nat", ""]), col
#         ] = pd.NA

#     realize_report = realize_report.dropna(subset=["FDBC_No"])

#     invoice_col = "Invoice_No" if "Invoice_No" in realize_report.columns else "Invoice"

#     def first_valid(values):
#         values = values.dropna()
#         return "" if values.empty else values.iloc[0]

#     onboard_range = (
#         realize_report.groupby("FDBC_No")
#         .agg(Min_Onboard=("OnBoard_Date", "min"), Max_Onboard=("OnBoard_Date", "max"))
#         .reset_index()
#     )

#     invoice_list = (
#         realize_report.dropna(subset=[invoice_col])
#         .groupby("FDBC_No")[invoice_col]
#         .apply(lambda s: ", ".join(sorted(s.astype(str).unique())))
#         .reset_index(name="INVOICES")
#     )

#     summary = (
#         realize_report.groupby("FDBC_No")
#         .agg(
#             Status=("Invoice_Status", first_valid),
#             Category=("Category", first_valid),
#             Buyer=("Buyer", first_valid),
#             Company=("LC_Factory", first_valid),
#             Bank=("Bank", first_valid),
#             PaymentTermXFactory=("Payment_After_ExFactory", "max"),
#             LC_Payment_Term=("LC_Payment_Term", first_valid),
#             ExFactoryDate=("ExFactory_Date", "max"),
#             BankSubmitDate=("BankSubmit_Date", "max"),
#             InvoiceDate=("Invoice_Date", "max"),
#             OriginalDueDate=("Due_Date", "max"),
#             Total_Invoices=(invoice_col, "nunique"),
#             Total_Invoice_Value=("Invoice_Value", "sum"),
#         )
#         .reset_index()
#     )

#     summary = summary.merge(onboard_range, on="FDBC_No", how="left")
#     summary = summary.merge(invoice_list, on="FDBC_No", how="left")

#     summary["IsBankSubmit?"] = np.where(summary["BankSubmitDate"].notna(), "YES", "NO")

#     summary["OnBoardDate(Range)"] = (
#         summary["Min_Onboard"].dt.strftime("%d-%b-%Y").fillna("")
#         + " to "
#         + summary["Max_Onboard"].dt.strftime("%d-%b-%Y").fillna("")
#     )
#     summary = summary.drop(columns=["Min_Onboard", "Max_Onboard"])

#     # Due date = ExFactory date + buyer payment term, falling back to the original due date
#     today = pd.Timestamp.today().normalize()
#     summary["Due_Date"] = summary["ExFactoryDate"] + pd.to_timedelta(
#         summary["PaymentTermXFactory"], unit="D"
#     )
#     summary["Due_Date"] = summary["Due_Date"].fillna(summary["OriginalDueDate"])

#     summary["IsFDBCRealizable?"] = np.where(
#         summary["Due_Date"] < today, "Need Realize", "Not Yet Realize"
#     )

#     summary = summary.sort_values("ExFactoryDate").reset_index(drop=True)
#     summary.insert(0, "SL#", range(1, len(summary) + 1))

#     summary = summary.rename(columns={
#         "FDBC_No": "Bank Ref No",
#         "PaymentTermXFactory": "Payment Term (X-Factory)",
#         "LC_Payment_Term": "LC Payment Term",
#         "Total_Invoice_Value": "Total Invoice Value",
#         "Due_Date": "Due Date",
#     })

#     return summary

In [ ]:
# def build_realize_pending_report(source_df):
#     """
#     Summarize 'Realize Pending' invoices by Bank Reference No (FDBC_No).

#     One output row is generated per FDBC_No, aggregating invoice information
#     and determining whether the bank reference should be realized based on:

#         (Today's Date - Bank Submit Date) > Payment Term (X-Factory)
#     """

#     realize_report = source_df[
#         source_df["Invoice_Status"].astype(str).str.strip().eq("Realize Pending")
#     ].copy()

#     # ------------------------------------------------------------------
#     # Convert date columns
#     # ------------------------------------------------------------------
#     date_cols = [
#         "Invoice_Date",
#         "Exp_Date",
#         "ExFactory_Date",
#         "OnBoard_Date",
#         "BankSubmit_Date",
#         "Realize_Date",
#     ]

#     for col in date_cols:
#         if col in realize_report.columns:
#             realize_report[col] = pd.to_datetime(
#                 realize_report[col],
#                 errors="coerce",
#                 dayfirst=True,
#             )

#     # ------------------------------------------------------------------
#     # Numeric Cleanup
#     # ------------------------------------------------------------------
#     def extract_number(series):
#         return pd.to_numeric(
#             series.astype(str).str.extract(r"(\d+\.?\d*)", expand=False),
#             errors="coerce",
#         )

#     realize_report["Invoice_Value"] = (
#         pd.to_numeric(realize_report["Invoice_Value"], errors="coerce")
#         .fillna(0)
#     )

#     realize_report["Payment_After_ExFactory"] = extract_number(
#         realize_report["Payment_After_ExFactory"]
#     )

#     # ------------------------------------------------------------------
#     # Normalize text fields
#     # ------------------------------------------------------------------
#     for col in ["LC_Payment_Term", "FDBC_No"]:
#         if col not in realize_report.columns:
#             continue

#         realize_report[col] = realize_report[col].astype("string").str.strip()

#         realize_report.loc[
#             realize_report[col].str.lower().isin(["", "nan", "none", "nat"]),
#             col,
#         ] = pd.NA

#     realize_report = realize_report.dropna(subset=["FDBC_No"])

#     invoice_col = (
#         "Invoice_No"
#         if "Invoice_No" in realize_report.columns
#         else "Invoice"
#     )

#     # ------------------------------------------------------------------
#     # Helper
#     # ------------------------------------------------------------------
#     def first_valid(series):
#         series = series.dropna()
#         return "" if series.empty else series.iloc[0]

#     # ------------------------------------------------------------------
#     # OnBoard Date Range
#     # ------------------------------------------------------------------
#     onboard_range = (
#         realize_report.groupby("FDBC_No")
#         .agg(
#             Min_Onboard=("OnBoard_Date", "min"),
#             Max_Onboard=("OnBoard_Date", "max"),
#         )
#         .reset_index()
#     )

#     # ------------------------------------------------------------------
#     # Invoice List
#     # ------------------------------------------------------------------
#     invoice_list = (
#         realize_report.dropna(subset=[invoice_col])
#         .groupby("FDBC_No")[invoice_col]
#         .apply(lambda s: ", ".join(sorted(s.astype(str).unique())))
#         .reset_index(name="INVOICES")
#     )

#     # ------------------------------------------------------------------
#     # Summary
#     # ------------------------------------------------------------------
#     summary = (
#         realize_report.groupby("FDBC_No")
#         .agg(
#             Status=("Invoice_Status", first_valid),
#             Category=("Category", first_valid),
#             Buyer=("Buyer", first_valid),
#             Company=("LC_Factory", first_valid),
#             Bank=("Bank", first_valid),
#             PaymentTermXFactory=("Payment_After_ExFactory", "max"),
#             LC_Payment_Term=("LC_Payment_Term", first_valid),
#             ExFactoryDate=("ExFactory_Date", "max"),
#             BankSubmitDate=("BankSubmit_Date", "max"),
#             InvoiceDate=("Invoice_Date", "max"),
#             Total_Invoices=(invoice_col, "nunique"),
#             Total_Invoice_Value=("Invoice_Value", "sum"),
#         )
#         .reset_index()
#     )

#     summary = summary.merge(onboard_range, on="FDBC_No", how="left")
#     summary = summary.merge(invoice_list, on="FDBC_No", how="left")

#     # ------------------------------------------------------------------
#     # Bank Submit Status
#     # ------------------------------------------------------------------
#     summary["IsBankSubmit?"] = np.where(
#         summary["BankSubmitDate"].notna(),
#         "YES",
#         "NO",
#     )

#     # ------------------------------------------------------------------
#     # OnBoard Date Range
#     # ------------------------------------------------------------------
#     summary["OnBoardDate(Range)"] = (
#         summary["Min_Onboard"].dt.strftime("%d-%b-%Y").fillna("")
#         + " to "
#         + summary["Max_Onboard"].dt.strftime("%d-%b-%Y").fillna("")
#     )

#     summary.drop(
#         columns=["Min_Onboard", "Max_Onboard"],
#         inplace=True,
#     )

#     # ------------------------------------------------------------------
#     # Realization Status
#     # ------------------------------------------------------------------
#     today = pd.Timestamp.today().normalize()

#     summary["Days Since Bank Submit"] = (
#         today - summary["BankSubmitDate"]
#     ).dt.days

#     summary["IsFDBCRealizable?"] = np.where(
#         (
#             summary["BankSubmitDate"].notna()
#         )
#         & (
#             summary["Days Since Bank Submit"]
#             > summary["PaymentTermXFactory"]
#         ),
#         "Need Realize",
#         "Not Yet Realize",
#     )

#     # ------------------------------------------------------------------
#     # Final Formatting
#     # ------------------------------------------------------------------
#     summary = summary.sort_values("ExFactoryDate").reset_index(drop=True)

#     summary.insert(0, "SL#", range(1, len(summary) + 1))

#     summary.rename(
#         columns={
#             "FDBC_No": "Bank Ref No",
#             "PaymentTermXFactory": "Payment Term (X-Factory)",
#             "LC_Payment_Term": "LC Payment Term",
#             "Total_Invoice_Value": "Total Invoice Value",
#         },
#         inplace=True,
#     )

#     return summary

In [ ]:
import numpy as np
import pandas as pd


def build_realize_pending_report(source_df):
    """
    Build a Realize Pending report grouped by FDBC_No.

    Realizability Rule:
        Days Since Bank Submit = Today - BankSubmit_Date

        If:
            Days Since Bank Submit > Payment Term (X-Factory)

        Then:
            Need Realize

        Otherwise:
            Not Yet Realize

    If BankSubmit_Date is missing:
        Not Yet Realize
    """

    # ============================================================
    # 1. Create a safe copy
    # ============================================================

    realize_report = source_df.copy()


    # ============================================================
    # 2. Filter Realize Pending invoices
    # ============================================================

    if "Invoice_Status" not in realize_report.columns:
        raise KeyError(
            "Required column 'Invoice_Status' was not found in the DataFrame."
        )

    realize_report = realize_report[
        realize_report["Invoice_Status"]
        .astype("string")
        .str.strip()
        .eq("Realize Pending")
    ].copy()


    # ============================================================
    # 3. Validate required columns
    # ============================================================

    required_columns = [
        "FDBC_No",
        "Invoice_Status",
        "Invoice_Value",
        "Payment_After_ExFactory",
        "OnBoard_Date",
        "BankSubmit_Date",
        "Invoice_Date",
        "ExFactory_Date",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in realize_report.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {missing_columns}"
        )


    # ============================================================
    # 4. Convert date columns safely
    # ============================================================

    date_columns = [
        "OnBoard_Date",
        "BankSubmit_Date",
        "Invoice_Date",
        "ExFactory_Date",
    ]

    for col in date_columns:

        realize_report[col] = pd.to_datetime(
            realize_report[col],
            errors="coerce",
            dayfirst=True
        )


    # ============================================================
    # 5. Numeric cleanup
    # ============================================================

    def extract_number(series):

        return pd.to_numeric(
            series.astype("string")
            .str.extract(r"(\d+(?:\.\d+)?)", expand=False),
            errors="coerce"
        )


    realize_report["Invoice_Value"] = pd.to_numeric(
        realize_report["Invoice_Value"],
        errors="coerce"
    ).fillna(0)


    realize_report["Payment_After_ExFactory"] = extract_number(
        realize_report["Payment_After_ExFactory"]
    )


    # ============================================================
    # 6. Normalize grouping columns
    # ============================================================

    for col in ["LC_Payment_Term", "FDBC_No"]:

        if col in realize_report.columns:

            realize_report[col] = (
                realize_report[col]
                .astype("string")
                .str.strip()
            )

            realize_report.loc[
                realize_report[col]
                .str.lower()
                .isin(["nan", "none", "nat", ""]),
                col
            ] = pd.NA


    # Remove records without FDBC number
    realize_report = realize_report.dropna(
        subset=["FDBC_No"]
    )


    # ============================================================
    # 7. Identify invoice column
    # ============================================================

    invoice_col = (
        "Invoice_No"
        if "Invoice_No" in realize_report.columns
        else "Invoice"
        if "Invoice" in realize_report.columns
        else None
    )

    if invoice_col is None:

        # Create a safe blank invoice column
        realize_report["Invoice_No"] = pd.NA
        invoice_col = "Invoice_No"


    # ============================================================
    # 8. Helper function
    # ============================================================

    def first_valid(series):

        series = series.dropna()

        if series.empty:
            return ""

        return series.iloc[0]


    # ============================================================
    # 9. On-board date range
    # ============================================================

    onboard_range = (

        realize_report
        .groupby("FDBC_No", dropna=False)
        .agg(
            Min_Onboard=("OnBoard_Date", "min"),
            Max_Onboard=("OnBoard_Date", "max")
        )
        .reset_index()
    )


    # ============================================================
    # 10. Invoice list
    # ============================================================

    invoice_list = (

        realize_report
        .dropna(subset=[invoice_col])
        .groupby("FDBC_No")[invoice_col]
        .apply(
            lambda s: ", ".join(
                sorted(
                    s.astype(str)
                    .str.strip()
                    .unique()
                )
            )
        )
        .reset_index(name="INVOICES")
    )


    # ============================================================
    # 11. Create summary
    # ============================================================

    aggregation_dict = {

        "Status": (
            "Invoice_Status",
            first_valid
        ),

        "Category": (
            "Category",
            first_valid
        ) if "Category" in realize_report.columns else (
            "FDBC_No",
            lambda x: ""
        ),

        "Buyer": (
            "Buyer",
            first_valid
        ) if "Buyer" in realize_report.columns else (
            "FDBC_No",
            lambda x: ""
        ),

        "Company": (
            "LC_Factory",
            first_valid
        ) if "LC_Factory" in realize_report.columns else (
            "FDBC_No",
            lambda x: ""
        ),

        "Bank": (
            "Bank",
            first_valid
        ) if "Bank" in realize_report.columns else (
            "FDBC_No",
            lambda x: ""
        ),

        "PaymentTermXFactory": (
            "Payment_After_ExFactory",
            "max"
        ),

        "LC_Payment_Term": (
            "LC_Payment_Term",
            first_valid
        ) if "LC_Payment_Term" in realize_report.columns else (
            "FDBC_No",
            lambda x: ""
        ),

        "ExFactoryDate": (
            "ExFactory_Date",
            "max"
        ),

        "BankSubmitDate": (
            "BankSubmit_Date",
            "max"
        ),

        "InvoiceDate": (
            "Invoice_Date",
            "max"
        ),

        "Total_Invoices": (
            invoice_col,
            "nunique"
        ),

        "Total_Invoice_Value": (
            "Invoice_Value",
            "sum"
        ),
    }


    summary = (

        realize_report
        .groupby("FDBC_No", dropna=False)
        .agg(**aggregation_dict)
        .reset_index()
    )


    # ============================================================
    # 12. Merge onboard date range and invoice list
    # ============================================================

    summary = summary.merge(
        onboard_range,
        on="FDBC_No",
        how="left"
    )

    summary = summary.merge(
        invoice_list,
        on="FDBC_No",
        how="left"
    )


    # ============================================================
    # 13. Bank submission status
    # ============================================================

    summary["IsBankSubmit?"] = np.where(
        summary["BankSubmitDate"].notna(),
        "YES",
        "NO"
    )


    # ============================================================
    # 14. Format onboard date range safely
    # ============================================================

    # Ensure datetime type before using .dt
    summary["Min_Onboard"] = pd.to_datetime(
        summary["Min_Onboard"],
        errors="coerce"
    )

    summary["Max_Onboard"] = pd.to_datetime(
        summary["Max_Onboard"],
        errors="coerce"
    )


    summary["OnBoardDate(Range)"] = (

        summary["Min_Onboard"]
        .dt.strftime("%d-%b-%Y")
        .fillna("")

        + " to "

        + summary["Max_Onboard"]
        .dt.strftime("%d-%b-%Y")
        .fillna("")
    )


    summary.drop(
        columns=["Min_Onboard", "Max_Onboard"],
        inplace=True
    )


    # ============================================================
    # 15. Calculate realizability
    # ============================================================

    today = pd.Timestamp.today().normalize()


    # Ensure BankSubmitDate is datetime
    summary["BankSubmitDate"] = pd.to_datetime(
        summary["BankSubmitDate"],
        errors="coerce"
    )


    days_since_submit = (

        today
        - summary["BankSubmitDate"]
    ).dt.days


    summary["IsFDBCRealizable?"] = np.where(

        summary["BankSubmitDate"].notna()

        & summary["PaymentTermXFactory"].notna()

        & (
            days_since_submit
            > summary["PaymentTermXFactory"]
        ),

        "Need Realize",

        "Not Yet Realize"
    )


    # ============================================================
    # 16. Sort data
    # ============================================================

    summary = summary.sort_values(
        by="ExFactoryDate",
        na_position="last"
    ).reset_index(drop=True)


    # ============================================================
    # 17. Add serial number
    # ============================================================

    summary.insert(
        0,
        "SL#",
        range(1, len(summary) + 1)
    )


    # ============================================================
    # 18. Rename columns
    # ============================================================

    summary.rename(
        columns={

            "FDBC_No": "Bank Ref No",

            "PaymentTermXFactory":
                "Payment Term (X-Factory)",

            "LC_Payment_Term":
                "LC Payment Term",

            "Total_Invoice_Value":
                "Total Invoice Value",
        },

        inplace=True
    )


    # ============================================================
    # 19. Final column order
    # ============================================================

    column_order = [

        "SL#",
        "Bank Ref No",
        "Status",
        "Category",
        "Buyer",
        "Company",
        "Bank",

        "Payment Term (X-Factory)",
        "LC Payment Term",

        "ExFactoryDate",
        "BankSubmitDate",
        "InvoiceDate",

        "Total_Invoices",
        "Total Invoice Value",

        "IsBankSubmit?",
        "OnBoardDate(Range)",
        "IsFDBCRealizable?",
        "INVOICES",
    ]


    # Keep only columns that actually exist
    final_columns = [

        col
        for col in column_order
        if col in summary.columns
    ]


    summary = summary[final_columns]


    return summary


# ============================================================
# Generate Realize Pending Report
# ============================================================

RealizePending = build_realize_pending_report(df)

print("Realize Pending Table Created Successfully")

RealizePending.head(3)

Realize Pending Table Created Successfully


,SL#,Bank Ref No,Status,Category,Buyer,Company,Bank,Payment Term (X-Factory),LC Payment Term,ExFactoryDate,BankSubmitDate,InvoiceDate,Total_Invoices,Total Invoice Value,IsBankSubmit?,OnBoardDate(Range),IsFDBCRealizable?,INVOICES
0,1,1647/23,Realize Pending,Sweater,DK Company,MGKFL,DBBL,177.0,75 Days,2023-04-18,2023-05-11,2023-04-18,1,4128.00,YES,30-Apr-2023 to 30-Apr-2023,Need Realize,2310036
1,2,FDBC/0471/23,Realize Pending,Sweater,Hellenic Trading AB,MFSL,ABL,80.0,At Sight,2023-04-21,2023-05-28,2023-04-20,1,42909.25,YES,22-May-2023 to 22-May-2023,Need Realize,2310069
2,3,FDBC/1221/23,Realize Pending,Sweater,Street One,MFSL,PRM,40.0,At Sight,2023-06-06,2023-06-21,2023-07-06,2,181585.28,YES,10-Jun-2023 to 16-Jun-2023,Need Realize,"2310863, 2310864"


## 13. Split Realize Pending by Owner / Category

In [ ]:
exclude_bank_refs = [
    "FDBC-278/24",
    "302/24",
    "0976/24",
    "0891/24",
    "0938/24",
    "040/24",
]

Tonni = RealizePending[
    (RealizePending["Company"] == "MGL")
    & (~RealizePending["Bank Ref No"].isin(exclude_bank_refs))
].copy()

Nowshin = RealizePending[RealizePending["Category"] == "Sweater"].copy()
Arif = RealizePending[RealizePending["Company"] == "MGSL"].copy()
Manzarul = RealizePending[RealizePending["Company"] == "MGNSL"].copy()


In [ ]:
RealizePending.to_excel(os.path.join(REALIZE_REPORTS_DIR, "RealizePending.xlsx"), index=False)
Nowshin.to_excel(os.path.join(REALIZE_REPORTS_DIR, "Nowshin.xlsx"), index=False)
Tonni.to_excel(os.path.join(REALIZE_REPORTS_DIR, "Tonni.xlsx"), index=False)
Arif.to_excel(os.path.join(REALIZE_REPORTS_DIR, "Arif.xlsx"), index=False)
Manzarul.to_excel(os.path.join(REALIZE_REPORTS_DIR, "Manzarul.xlsx"), index=False)

## 14. On Board Pending Report

In [ ]:
# def build_onboard_pending_report(source_df):
#     """Invoices still awaiting on-board status, 15+ days past ex-factory.

#     Note: unlike the ExFactory/BankSubmit reports below, this report does
#     not regenerate the 'SL#' serial column — it is carried over as-is from
#     the master table.
#     """
#     source_df["ExFactory_Date"] = pd.to_datetime(source_df["ExFactory_Date"], errors="coerce")
#     today = pd.Timestamp.today().normalize()

#     source_df["Invoice_Date"] = pd.to_datetime(source_df["Invoice_Date"], errors="coerce")
#     today = pd.Timestamp.today().normalize()

#     source_df["Invoice_Date"] = pd.to_datetime(source_df["Invoice_Date"], errors="coerce")
#     today = pd.Timestamp.today().normalize()

    

#     onboard_pending = source_df[
#         (source_df["Invoice_Status"] == "On Board Pending")
#         & ((today - source_df["ExFactory_Date"]).dt.days >= 15)
#     ].copy()

#     columns_to_drop = [
#         "LC_No", "LC_Value", "LC_Qty", "LC_Payment_Term", "Payment_After_ExFactory",
#         "Merchandiser", "Port", "Ctn_Qty", "Dozen", "MVessel_No", "Due_Date",
#         "FDBC_No", "BankSubmit_Date", "Is_FDBC?", "BL_No", "BL_Date", "FCR_No",
#         "FCR_Date", "Carton_Qty", "Discount", "Commission", "Shipping_Bill_No",
#         "Shipping_Bill_Date", "Realize_No", "Realize_Date", "Realize_Value", "Order_No",
#     ]
#     onboard_pending = onboard_pending.drop(columns=columns_to_drop, errors="ignore")
#     onboard_pending = onboard_pending.reset_index(drop=True)

#     return onboard_pending

In [ ]:
import pandas as pd


def build_onboard_pending_report(source_df):
    """Invoices still awaiting on-board status, 15+ days past ex-factory.

    Note: unlike the ExFactory/BankSubmit reports below, this report does
    not regenerate the 'SL#' serial column — it is carried over as-is from
    the master table.

    Date columns are kept as raw pandas Timestamps (e.g. 2023-04-05 00:00:00)
    with no string formatting applied.
    """
    # Parse all relevant date columns once — kept as datetime (no strftime)
    date_cols = ["ExFactory_Date", "Invoice_Date"]
    for col in date_cols:
        if col in source_df.columns:
            source_df[col] = pd.to_datetime(source_df[col], errors="coerce")

    today = pd.Timestamp.today().normalize()

    onboard_pending = source_df[
        (source_df["Invoice_Status"] == "On Board Pending")
        & ((today - source_df["ExFactory_Date"]).dt.days >= 15)
    ].copy()

    columns_to_drop = [
        "LC_No", "LC_Value", "LC_Qty", "LC_Payment_Term", "Payment_After_ExFactory",
        "Merchandiser", "Port", "Ctn_Qty", "Dozen", "MVessel_No", "Due_Date",
        "FDBC_No", "BankSubmit_Date", "Is_FDBC?", "BL_No", "BL_Date", "FCR_No",
        "FCR_Date", "Carton_Qty", "Discount", "Commission", "Shipping_Bill_No",
        "Shipping_Bill_Date", "Realize_No", "Realize_Date", "Realize_Value", "Order_No",
    ]
    onboard_pending = onboard_pending.drop(columns=columns_to_drop, errors="ignore")
    onboard_pending = onboard_pending.reset_index(drop=True)

    return onboard_pending

In [ ]:
OnBoardPending = build_onboard_pending_report(df)
OnBoardPending.sample(3)

,SL#,Category,LC_Factory,Invoice_No,Invoice_Date,Invoice_Value,Invoice_Qty,Exp_No,Exp_Date,Buyer,...,ExFactory,ExFactory_Date,ExFactory_Qty,Last_ExFactory_Date,Shipping_Mode,OnBoard_Date,Bank,Invoice_Status,Is_Realize?,FVessel_No
86,45100,Woven,MGL,2608945,2026-03-08,4561.75,1285,00001880-001461-2026,04-08-2026,EL CORTE INGLES,...,MGNFL-2,2026-03-08,1285,03-08-2026,Sea,,OBL,On Board Pending,No,NaN
46,25681,Woven,MGNSL,2609032,2026-04-08,25886.07,4460,00001079-003068-2026,04-08-2026,Best Seller,...,MGNSL,2026-04-08,4460,04-08-2026,Sea,,Al Arafah,On Board Pending,No,NaN
41,21632,Woven,MGSL,2408423,2024-02-06,178.89,67,00000042-013422-2024,02-06-2024,H&M,...,MGSL,2024-03-06,67,03-06-2024,Sea,,ABL,On Board Pending,No,NaN


In [ ]:
OnBoardPending.to_excel(os.path.join(OUTPUT_DIR, "OnBoardPending.xlsx"), index=False)

## 15. Ex Factory Pending Report

In [ ]:
def build_exfactory_pending_report(source_df):
    """Invoices still awaiting ex-factory shipment, 7+ days past invoice date."""
    exfactory_pending = source_df.copy()

    # date_cols = [
    #     "Invoice_Date", "Exp_Date", "ExFactory_Date", "OnBoard_Date", "Due_Date",
    #     "BankSubmit_Date", "BL_Date", "FCR_Date", "Shipping_Bill_Date", "Realize_Date",
    # ]
    # for col in date_cols:
    #     if col in exfactory_pending.columns:
    #         exfactory_pending[col] = pd.to_datetime(
    #             exfactory_pending[col], errors="coerce", dayfirst=True
    #         ).dt.normalize()

    today = pd.Timestamp.today().normalize()
    mask = (
        exfactory_pending["Invoice_Status"].astype(str).str.strip().eq("ExFactory Pending")
        & exfactory_pending["Invoice_Date"].notna()
        & ((today - exfactory_pending["Invoice_Date"]).dt.days >= 7)
    )
    exfactory_pending = exfactory_pending.loc[mask].copy()

    columns_to_drop = [
        "LC_No", "LC_Value", "LC_Qty", "LC_Payment_Term", "Payment_After_ExFactory",
        "Merchandiser", "Port", "Ctn_Qty", "Dozen", "MVessel_No", "Due_Date",
        "FDBC_No", "BankSubmit_Date", "Is_FDBC?", "BL_No", "BL_Date", "FCR_No",
        "FCR_Date", "Carton_Qty", "Discount", "Commission", "Shipping_Bill_No",
        "Shipping_Bill_Date", "Realize_No", "Realize_Date", "Realize_Value",
        "Order_No", "ExFactory_No", "ExFactory",
    ]
    exfactory_pending = exfactory_pending.drop(columns=columns_to_drop, errors="ignore")

    if "Invoice_Date" in exfactory_pending.columns:
        exfactory_pending = exfactory_pending.sort_values(
            by="Invoice_Date", ascending=True, na_position="last"
        )
    exfactory_pending = exfactory_pending.reset_index(drop=True)

    # Regenerate the serial column
    if "SL#" in exfactory_pending.columns:
        exfactory_pending = exfactory_pending.drop(columns=["SL#"])
    exfactory_pending.insert(0, "SL#", range(1, len(exfactory_pending) + 1))

    # Clean text columns only (dates are kept as real datetimes)
    text_cols = exfactory_pending.select_dtypes(include=["object", "string"]).columns
    exfactory_pending[text_cols] = (
        exfactory_pending[text_cols]
        .replace(["nan", "NaN", "NaT", "None", "none"], "")
        .fillna("")
    )

    return exfactory_pending

In [ ]:
ExFactoryPending = build_exfactory_pending_report(df)

print("ExFactory Pending Table Created Successfully")
print(f"Total Rows: {len(ExFactoryPending)}")
ExFactoryPending.head(3)

ExFactory Pending Table Created Successfully
Total Rows: 26


,SL#,Category,LC_Factory,Invoice_No,Invoice_Date,Invoice_Value,Invoice_Qty,Exp_No,Exp_Date,Buyer,ExFactory_Date,ExFactory_Qty,Last_ExFactory_Date,Shipping_Mode,OnBoard_Date,Bank,Invoice_Status,Is_Realize?,FVessel_No
0,1,Woven,MGSL,MGSL/23/3395,2023-04-05,384.30,122,006091,04-05-2023,H&M,1900-01-01,<NA>,,Sea,,ABL,ExFactory Pending,No,
1,2,Woven,MGSL,MGSL/23/3429,2023-06-05,4460.72,814,0042/006138/23,07-05-2023,H&M,1900-01-01,<NA>,,Sea,13-05-2023,ABL,ExFactory Pending,Yes,
2,3,Woven,MGNSL,2311849,2023-06-08,149.10,34,,06-08-2023,HAGGAR CLOTHING CO.,1900-01-01,<NA>,,Air,10-08-2023,ABL,ExFactory Pending,No,


In [ ]:
ExFactoryPending.to_excel(os.path.join(OUTPUT_DIR, "ExFactoryPending.xlsx"), index=False)

## 16. Bank Submit Pending Report

In [ ]:
def build_bank_submit_pending_report(source_df):
    """Invoices still awaiting bank submission, 7+ days past invoice date."""
    bank_submit_pending = source_df.copy()

    # date_cols = [
    #     "Invoice_Date", "Exp_Date", "ExFactory_Date", "OnBoard_Date", "Due_Date",
    #     "BankSubmit_Date", "BL_Date", "FCR_Date", "Shipping_Bill_Date", "Realize_Date",
    # ]
    # for col in date_cols:
    #     if col in bank_submit_pending.columns:
    #         bank_submit_pending[col] = pd.to_datetime(
    #             bank_submit_pending[col], errors="coerce", dayfirst=True
    #         ).dt.normalize()

    today = pd.Timestamp.today().normalize()
    mask = (
        bank_submit_pending["Invoice_Status"].astype(str).str.strip().eq("Bank Submit Pending")
        & bank_submit_pending["Invoice_Date"].notna()
        & ((today - bank_submit_pending["Invoice_Date"]).dt.days >= 7)
    )
    bank_submit_pending = bank_submit_pending.loc[mask].copy()

    columns_to_drop = [
        "LC_No", "LC_Value", "LC_Qty", "LC_Payment_Term", "Payment_After_ExFactory",
        "Merchandiser", "Port", "Ctn_Qty", "Dozen", "MVessel_No", "Due_Date",
        "FDBC_No", "BankSubmit_Date", "Is_FDBC?", "BL_No", "BL_Date", "FCR_No",
        "FCR_Date", "Carton_Qty", "Discount", "Commission", "Shipping_Bill_No",
        "Shipping_Bill_Date", "Realize_No", "Realize_Date", "Realize_Value",
        "Order_No", "ExFactory_No", "ExFactory",
    ]
    bank_submit_pending = bank_submit_pending.drop(columns=columns_to_drop, errors="ignore")

    if "Invoice_Date" in bank_submit_pending.columns:
        bank_submit_pending = bank_submit_pending.sort_values(
            by="Invoice_Date", ascending=True, na_position="last"
        )
    bank_submit_pending = bank_submit_pending.reset_index(drop=True)

    # Regenerate the serial column
    if "SL#" in bank_submit_pending.columns:
        bank_submit_pending = bank_submit_pending.drop(columns=["SL#"])
    bank_submit_pending.insert(0, "SL#", range(1, len(bank_submit_pending) + 1))

    # Clean text columns only (dates are kept as real datetimes)
    text_cols = bank_submit_pending.select_dtypes(include=["object", "string"]).columns
    bank_submit_pending[text_cols] = (
        bank_submit_pending[text_cols]
        .replace(["nan", "NaN", "NaT", "None", "none"], "")
        .fillna("")
    )

    return bank_submit_pending

In [ ]:
BankSubmitPending = build_bank_submit_pending_report(df)

print("BankSubmit Pending Table Created Successfully")
print(f"Total Rows: {len(BankSubmitPending)}")
BankSubmitPending.head(3)

BankSubmit Pending Table Created Successfully
Total Rows: 689


,SL#,Category,LC_Factory,Invoice_No,Invoice_Date,Invoice_Value,Invoice_Qty,Exp_No,Exp_Date,Buyer,ExFactory_Date,ExFactory_Qty,Last_ExFactory_Date,Shipping_Mode,OnBoard_Date,Bank,Invoice_Status,Is_Realize?,FVessel_No
0,1,Woven,MGSL,MGSL/23/1893,2023-01-03,30004.20,9495,003274,02-03-2023,H&M,2023-09-03,9495,09-03-2023,Sea,08-03-2023,ABL,Bank Submit Pending,No,
1,2,Woven,MGSL,MGSL/23/2540,2023-01-04,5263.36,2056,004569,01-04-2023,H&M,2023-02-04,2056,02-04-2023,Sea,08-04-2023,ABL,Bank Submit Pending,No,
2,3,Woven,MGSL,MGSL/23/2701,2023-01-04,2679.00,570,004795,01-04-2023,H&M,2023-02-04,570,02-04-2023,Sea,09-04-2023,ABL,Bank Submit Pending,No,


In [ ]:
BankSubmitPending.to_excel(os.path.join(OUTPUT_DIR, "BankSubmitPending.xlsx"), index=False)

## 17. Data Validation

In [ ]:
# """
# ===============================================================================
# Invoice Validation & Cancelled Invoice Detection
# ===============================================================================

# Author : Your Name
# Purpose:
#     Perform business-rule validation on export invoice data and identify
#     cancelled invoices.

# Input:
#     Pandas DataFrame

# Output:
#     DataFrame with validation columns appended.

# ===============================================================================
# """

# import os
# from pathlib import Path

# import numpy as np
# import pandas as pd


# # =============================================================================
# # Configuration
# # =============================================================================

# OUTPUT_FILE = "EIS_Validated.xlsx"


# # =============================================================================
# # Helper Functions
# # =============================================================================

# def is_blank(series: pd.Series) -> pd.Series:
#     """
#     Returns True where values are blank or NaN.
#     """
#     return series.fillna("").astype(str).str.strip().eq("")


# def validate_date_order(start_date: pd.Series,
#                         end_date: pd.Series) -> pd.Series:
#     """
#     Validate that start_date <= end_date.

#     Returns
#     -------
#     Series
#         "OK"
#         "Wrong"
#         ""
#     """

#     return np.where(
#         start_date.isna() | end_date.isna(),
#         "",
#         np.where(start_date <= end_date, "OK", "Wrong")
#     )


# def calculate_gap(start_date: pd.Series,
#                   end_date: pd.Series) -> pd.Series:
#     """
#     Calculate day difference between two dates.
#     """

#     return np.where(
#         start_date.isna() | end_date.isna(),
#         np.nan,
#         (end_date - start_date).dt.days
#     )


# # =============================================================================
# # Main Validation Function
# # =============================================================================

# def validate_invoice_data(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Validate invoice information and identify cancelled invoices.

#     Parameters
#     ----------
#     df : pandas.DataFrame

#     Returns
#     -------
#     pandas.DataFrame
#     """

#     result = df.copy()

#     # -------------------------------------------------------------------------
#     # 1. Invoice Date vs ExFactory Date
#     # -------------------------------------------------------------------------

#     result["WrongExFactoryDate"] = validate_date_order(
#         result["Invoice_Date"],
#         result["ExFactory_Date"]
#     )

#     result["InvDateExDateGap"] = calculate_gap(
#         result["Invoice_Date"],
#         result["ExFactory_Date"]
#     )

#     # -------------------------------------------------------------------------
#     # 2. ExFactory Date vs OnBoard Date
#     # -------------------------------------------------------------------------

#     result["WrongOnBoardDate"] = validate_date_order(
#         result["ExFactory_Date"],
#         result["OnBoard_Date"]
#     )

#     result["ExDateOnDateGap"] = calculate_gap(
#         result["ExFactory_Date"],
#         result["OnBoard_Date"]
#     )

#     # -------------------------------------------------------------------------
#     # 3. EXP Date Validation
#     # -------------------------------------------------------------------------

#     result["WrongEXPDate"] = np.where(

#         result["Exp_Date"].isna()
#         | result["Invoice_Date"].isna()
#         | result["OnBoard_Date"].isna(),

#         "",

#         np.where(

#             (result["Exp_Date"] >= result["Invoice_Date"])
#             &
#             (result["Exp_Date"] <= result["OnBoard_Date"]),

#             "OK",
#             "Wrong"
#         )
#     )

#     # -------------------------------------------------------------------------
#     # 4. BL Date vs Bank Submit Date
#     # -------------------------------------------------------------------------

#     if "BL_Date" in result.columns:

#         result["WrongBankSubmitDate"] = validate_date_order(
#             result["BL_Date"],
#             result["BankSubmit_Date"]
#         )

#     else:

#         result["WrongBankSubmitDate"] = ""

#     # -------------------------------------------------------------------------
#     # 5. Cancelled Invoice Detection
#     # -------------------------------------------------------------------------

#     cutoff_date = pd.Timestamp.today().normalize() - pd.DateOffset(months=5)

#     cancelled = (

#         (result["Invoice_Date"] <= cutoff_date)

#         & result["ExFactory_Date"].isna()

#         & is_blank(result["ExFactory_No"])

#         & result["OnBoard_Date"].isna()

#         & is_blank(result["FDBC_No"])

#         & result["BankSubmit_Date"].isna()

#         & is_blank(result["Realize_No"])

#         & result["Realize_Date"].isna()

#         & (
#             result["Realize_Value"].fillna(0).eq(0)
#         )

#         & (
#             result["Is_FDBC?"]
#             .fillna("")
#             .astype(str)
#             .str.strip()
#             .str.upper()
#             .eq("NO")
#         )

#     )

#     result["CancelledInvoice"] = np.where(
#         cancelled,
#         "YES",
#         "Waiting"
#     )

#     return result


# # =============================================================================
# # Summary Function
# # =============================================================================

# def print_summary(df: pd.DataFrame):
#     """
#     Print validation statistics.
#     """

#     print("=" * 70)
#     print("VALIDATION SUMMARY")
#     print("=" * 70)

#     print(f"Total Records              : {len(df):,}")
#     print(f"Cancelled Invoices         : {(df['CancelledInvoice']=='YES').sum():,}")
#     print(f"Waiting Invoices           : {(df['CancelledInvoice']=='Waiting').sum():,}")

#     print("-" * 70)

#     print(f"Wrong ExFactory Date       : {(df['WrongExFactoryDate']=='Wrong').sum():,}")
#     print(f"Wrong OnBoard Date         : {(df['WrongOnBoardDate']=='Wrong').sum():,}")
#     print(f"Wrong EXP Date             : {(df['WrongEXPDate']=='Wrong').sum():,}")
#     print(f"Wrong BankSubmit Date      : {(df['WrongBankSubmitDate']=='Wrong').sum():,}")

#     print("=" * 70)


# # =============================================================================
# # Execution
# # =============================================================================

# if __name__ == "__main__":

#     # df should already be loaded
#     validated_df = validate_invoice_data(df)

#     print_summary(validated_df)

#     output_path = Path(OUTPUT_FILE)

#     validated_df.to_excel(output_path, index=False)

#     print(f"\nValidation file saved to:\n{output_path.resolve()}")

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta


def validate_and_extract_invoices(df):
    """
    Validate invoice data and identify cancelled invoices.

    All date columns are safely converted to datetime before:
    - Date comparison
    - Date subtraction
    - Date difference calculation
    """

    # ============================================================
    # 1. Create a safe copy
    # ============================================================

    result_df = df.copy()


    # ============================================================
    # 2. Convert all required date columns to datetime
    # ============================================================

    date_columns = [
        "Invoice_Date",
        "ExFactory_Date",
        "OnBoard_Date",
        "Exp_Date",
        "BL_Date",
        "BankSubmit_Date",
        "Realize_Date",
    ]

    for col in date_columns:

        if col in result_df.columns:

            result_df[col] = pd.to_datetime(
                result_df[col],
                errors="coerce",
                dayfirst=True
            )


    # ============================================================
    # 3. Wrong ExFactory Date
    # ============================================================
    # Rule:
    # Invoice_Date <= ExFactory_Date

    result_df["WrongExFactoryDate"] = np.select(

        [
            result_df["Invoice_Date"].isna()
            | result_df["ExFactory_Date"].isna(),

            result_df["Invoice_Date"]
            <= result_df["ExFactory_Date"]
        ],

        [
            "",

            "OK"
        ],

        default="Wrong"
    )


    # ============================================================
    # 4. Invoice Date to ExFactory Date Gap
    # ============================================================

    result_df["InvDateExDateGap"] = (

        result_df["ExFactory_Date"]
        - result_df["Invoice_Date"]
    ).dt.days


    # Convert missing gap to blank
    result_df["InvDateExDateGap"] = (
        result_df["InvDateExDateGap"]
        .astype("Int64")
    )


    # ============================================================
    # 5. Wrong OnBoard Date
    # ============================================================
    # Rule:
    # ExFactory_Date <= OnBoard_Date

    result_df["WrongOnBoardDate"] = np.select(

        [
            result_df["ExFactory_Date"].isna()
            | result_df["OnBoard_Date"].isna(),

            result_df["ExFactory_Date"]
            <= result_df["OnBoard_Date"]
        ],

        [
            "",

            "OK"
        ],

        default="Wrong"
    )


    # ============================================================
    # 6. ExFactory Date to OnBoard Date Gap
    # ============================================================

    result_df["ExDateOnDateGap"] = (

        result_df["OnBoard_Date"]
        - result_df["ExFactory_Date"]
    ).dt.days


    result_df["ExDateOnDateGap"] = (
        result_df["ExDateOnDateGap"]
        .astype("Int64")
    )


    # ============================================================
    # 7. Wrong EXP Date
    # ============================================================
    # Rule:
    # Invoice_Date <= Exp_Date <= OnBoard_Date

    result_df["WrongEXPDate"] = np.select(

        [
            result_df["Exp_Date"].isna()
            | result_df["Invoice_Date"].isna()
            | result_df["OnBoard_Date"].isna(),

            (
                (result_df["Exp_Date"]
                 >= result_df["Invoice_Date"])
                &
                (result_df["Exp_Date"]
                 <= result_df["OnBoard_Date"])
            )
        ],

        [
            "",

            "OK"
        ],

        default="Wrong"
    )


    # ============================================================
    # 8. Wrong Bank Submit Date
    # ============================================================
    # Rule:
    # BL_Date <= BankSubmit_Date

    result_df["WrongBankSubmitDate"] = np.select(

        [
            result_df["BL_Date"].isna()
            | result_df["BankSubmit_Date"].isna(),

            result_df["BL_Date"]
            <= result_df["BankSubmit_Date"]
        ],

        [
            "",

            "OK"
        ],

        default="Wrong"
    )


    # ============================================================
    # 9. Cancelled Invoice Detection
    # ============================================================

    today = pd.Timestamp.today().normalize()

    five_months_ago = (
        today
        - timedelta(days=150)
    )


    # Invoice must be at least 5 months old
    invoice_date_check = (

        result_df["Invoice_Date"]
        <= five_months_ago
    )


    # Empty value helper
    def is_blank(series):

        return (

            series.isna()
            |
            series.astype("string")
            .str.strip()
            .str.lower()
            .isin(["", "nan", "none", "nat"])
        )


    # Cancellation conditions

    exfactory_date_blank = (
        result_df["ExFactory_Date"].isna()
    )

    exfactory_no_blank = (
        is_blank(result_df["ExFactory_No"])
    )

    onboard_date_blank = (
        result_df["OnBoard_Date"].isna()
    )

    fdbc_no_blank = (
        is_blank(result_df["FDBC_No"])
    )

    banksubmit_date_blank = (
        result_df["BankSubmit_Date"].isna()
    )

    realize_no_blank = (
        is_blank(result_df["Realize_No"])
    )

    realize_date_blank = (
        result_df["Realize_Date"].isna()
    )


    # Safely convert Realize_Value to numeric
    realize_value_numeric = pd.to_numeric(
        result_df["Realize_Value"],
        errors="coerce"
    )


    realize_value_blank_or_zero = (

        result_df["Realize_Value"].isna()

        |

        realize_value_numeric.eq(0)
    )


    is_fdbc_no = (

        result_df["Is_FDBC?"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("no")
    )


    # All conditions must be TRUE
    is_cancelled = (

        invoice_date_check

        & exfactory_date_blank

        & exfactory_no_blank

        & onboard_date_blank

        & fdbc_no_blank

        & banksubmit_date_blank

        & realize_no_blank

        & realize_date_blank

        & realize_value_blank_or_zero

        & is_fdbc_no
    )


    result_df["CancelledInvoice"] = np.where(

        is_cancelled,

        "YES",

        "Waiting"
    )


    # ============================================================
    # 10. Extract Required Columns
    # ============================================================

    extracted_columns = [

        "SL#",
        "Category",
        "LC_Factory",
        "LC_No",
        "Payment_After_ExFactory",

        "Invoice_No",
        "Invoice_Date",

        "Exp_No",
        "Exp_Date",

        "Buyer",
        "Merchandiser",

        "ExFactory_Date",
        "ExFactory_No",
        "ExFactory",

        "OnBoard_Date",

        "Shipping_Bill_No",
        "Shipping_Bill_Date",

        "Bank",
        "FDBC_No",

        "BankSubmit_Date",
        "Is_FDBC?",

        "Invoice_Status",

        "Realize_No",
        "Realize_Date",
        "Realize_Value",
        "Is_Realize?",

        "WrongExFactoryDate",
        "InvDateExDateGap",

        "WrongOnBoardDate",
        "ExDateOnDateGap",

        "WrongEXPDate",
        "WrongBankSubmitDate",

        "CancelledInvoice",
    ]


    # Keep only columns that exist
    existing_columns = [

        col
        for col in extracted_columns
        if col in result_df.columns
    ]


    validation_report = (
        result_df[existing_columns]
        .copy()
    )


    return validation_report



In [ ]:
validation_report = validate_and_extract_invoices(df)

print("\n" + "=" * 80)
print("DATA VALIDATION & CANCELLED INVOICE REPORT")
print("=" * 80)

print(f"\nTotal Records: {len(validation_report)}")

print(
    f"Cancelled Invoices: "
    f"{(validation_report['CancelledInvoice'] == 'YES').sum()}"
)

print(
    f"Waiting/Active Invoices: "
    f"{(validation_report['CancelledInvoice'] == 'Waiting').sum()}"
)


DATA VALIDATION & CANCELLED INVOICE REPORT

Total Records: 51325
Cancelled Invoices: 0
Waiting/Active Invoices: 51325
